# Fully plastic cortical column: feature-map and kernel sweep

Nine fixed saccades (3×3), fresh learned kernels for every run, reciprocal apical STDP, and centered R-STDP decisions. All computation runs on one CoNeX/PymoNNtorch graph. No pretrained features or offline feature cache is used. CUDA is required for the full benchmark.

This notebook calls the canonical training pipeline; it does not maintain a second model implementation.

In [ ]:
from pathlib import Path
import itertools, json, csv, gc
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'column/Column.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from column import Column, Config, L4, L23, L56, DecisionLayer
from column.train import run
from column.data.config import Config as DataConfig
from column.data.pipeline import make_caltech_loaders, make_mnist_loaders
assert torch.cuda.is_available(), 'Install a CUDA-enabled PyTorch build for this experiment.'


## Settings

`FEATURE_COUNTS` controls the number of learned L4 kernels; `KERNEL_SIZES` controls their spatial size. `POOL_SIDES` controls the pooled feature-map size, giving `features × pool_side²` L2/3 neurons. For a kernel sweep, set `KERNEL_SIZES = [7, 11, 15]`. Every combination starts from new random weights. The number and locations of saccades stay fixed.

Use 20 epochs to match the fresh Caltech validation run. Shortened runs are smoke tests, not accuracy comparisons. MNIST uses 20,000 training and 5,000 validation examples; this is not the official test set.

In [ ]:
DATASET = 'caltech'
FEATURE_COUNTS = [25] if DATASET == 'caltech' else [100]
KERNEL_SIZES = [11]
POOL_SIDES = [3, 5, 7]
SEEDS = [42]
EPOCHS = 20
OUTPUT = ROOT / 'outputs/feature_sweep'
OUTPUT.mkdir(parents=True, exist_ok=True)


## Data and model

The canonical runner uses the same deterministic split and DoG preprocessing. It loads images, constructs a fresh column, initializes its native network, trains every plastic pathway, and evaluates with all plasticity disabled. Labels are used for the authorized training-only reference-frame cue and for an external scalar reward, never as a decision teacher current.

In [ ]:
data_config = DataConfig()
data_config.paths.caltech_path = str(ROOT / 'data/caltech')
data_config.paths.mnist_path = str(ROOT / 'data/mnist')
factory = make_caltech_loaders if DATASET == 'caltech' else make_mnist_loaders
train_loader, validation_loader = factory(data_config, seed=42)
print('Available split:', len(train_loader.dataset), 'training,', len(validation_loader.dataset), 'validation')
print('All training stages enabled: L4, reciprocal bindings, decision R-STDP')


## Full training

Each run returns the final model and measured validation results. Selection is by the configured final epoch, not by the best validation epoch. Checkpoints, predictions, labels and source hashes are written for every run.

In [ ]:
rows = []
for maps, kernel, pool, seed in itertools.product(FEATURE_COUNTS, KERNEL_SIZES, POOL_SIDES, SEEDS):
    name = f'{DATASET}_maps{maps}_kernel{kernel}_pool{pool}_seed{seed}'
    model, report = run(dataset=DATASET, epochs=EPOCHS, seed=seed, maps=maps,
                        kernel=kernel, pool=pool, output=str(OUTPUT / name))
    rows.append(dict(dataset=DATASET, maps=maps, kernel=kernel, pool=pool, seed=seed,
                     l23_neurons=maps * pool**2, accuracy=report['final_accuracy'],
                     n_validation=report['n_validation'], checkpoint=str(OUTPUT/name/'model.pt')))
    del model
    gc.collect()
    torch.cuda.empty_cache()
with (OUTPUT/'summary.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader(); writer.writerows(rows)
rows


## Results and visualization

Points show mean final accuracy across configured seeds; error bars show their standard deviation. A single seed produces zero error bars and is not an uncertainty estimate.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for maps, kernel in itertools.product(FEATURE_COUNTS, KERNEL_SIZES):
    means, errors = [], []
    for pool in POOL_SIDES:
        values = [100*r['accuracy'] for r in rows if (r['maps'],r['kernel'],r['pool']) == (maps,kernel,pool)]
        means.append(np.mean(values)); errors.append(np.std(values, ddof=1) if len(values)>1 else 0.)
    ax.errorbar(POOL_SIDES, means, yerr=errors, marker='o', capsize=3,
                label=f'{maps} features; {kernel}×{kernel} kernel')
ax.set(xlabel='Pooled feature-map side', ylabel='Validation accuracy (%)', ylim=(0, 101))
ax.set_xticks(POOL_SIDES)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT/'feature_map_accuracy.pdf', bbox_inches='tight')
fig.savefig(OUTPUT/'feature_map_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()
print('Editable values:', OUTPUT/'summary.csv')


## Loading a trained model or selecting stages

`Column.load(checkpoint)` restores the model. `model.plasticity()` enables all plastic pathways; `model.plasticity(('l4',))`, `('bindings',)` or `('decision',)` selects a stage. The sweep above always enables all stages. The decision rule is centered local R-STDP, not GatedRSTDP. Biological simplifications, event timing and benchmark provenance are documented in the repository README.